In [1]:
from neem_interface_python.designator_decorators import init_neem
from pycram.worlds.bullet_world import BulletWorld
from pycram.designators.action_designator import *
from pycram.designators.location_designator import *
from pycram.designators.object_designator import *
from pycram.datastructures.enums import WorldMode
from pycram.datastructures.pose import Pose
from pycram.process_module import simulated_robot, with_simulated_robot
from pycram.object_descriptors.urdf import ObjectDescription
from pycram.world_concepts.world_object import Object
from pycrap import Robot, Apartment, Milk, Cereal, Spoon, Bowl
import numpy as np
from pycram.tasktree import *



[INFO] [1740042757.945965]: [KnowRob] initialize client...
[INFO] [1740042757.948618]: [KnowRob]  done.


In [2]:
world = BulletWorld(WorldMode.GUI)
extension = ObjectDescription.get_file_extension()
robot = Object("pr2", Robot, f"pr2{extension}", pose=Pose([1, 2, 0]))
apartment = Object("apartment", Apartment, f"apartment{extension}")
robot_desig = BelieveObject(names=["pr2"])
apartment_desig = BelieveObject(names=["apartment"])

[INFO] [1740042768.312989]: [cache_manager.py:105:look_for_file_in_data_dir] Found file plane.urdf in /home/hawkin/ros/pycram_ws/src/pycram/resources/objects/plane.urdf
[INFO] [1740042768.400941]: [cache_manager.py:105:look_for_file_in_data_dir] Found file pr2.urdf in /home/hawkin/ros/pycram_ws/src/pycram/resources/robots/pr2.urdf
[INFO] [1740042771.709493]: [cache_manager.py:105:look_for_file_in_data_dir] Found file apartment.urdf in /home/hawkin/ros/pycram_ws/src/pycram/resources/objects/apartment.urdf


Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


In [3]:
def setup_world():
    global apartment
    np.random.seed(420)
    milk = Object("milk", Milk, "milk.stl", pose=Pose([2.5, 2, 1.02]), color=Color(1, 0, 0, 1))
    cereal = Object("cereal", Cereal, "breakfast_cereal.stl", pose=Pose([2.5, 2.3, 1.05]), color=Color(0, 1, 0, 1))
    spoon = Object("spoon", Spoon, "spoon.stl", pose=Pose([2.4, 2.24, 0.85]), color=Color(0, 0, 1, 1))
    bowl = Object("bowl", Bowl, "bowl.stl", pose=Pose([2.5, 2.2, 1.02]), color=Color(1, 1, 0, 1))
    apartment.attach(spoon, 'cabinet10_drawer_top')

setup_world()

[INFO] [1740042781.428030]: [cache_manager.py:105:look_for_file_in_data_dir] Found file milk.stl in /home/hawkin/ros/pycram_ws/src/pycram/resources/objects/milk.stl
[INFO] [1740042781.522669]: [cache_manager.py:105:look_for_file_in_data_dir] Found file breakfast_cereal.stl in /home/hawkin/ros/pycram_ws/src/pycram/resources/objects/breakfast_cereal.stl
[INFO] [1740042781.622743]: [cache_manager.py:105:look_for_file_in_data_dir] Found file spoon.stl in /home/hawkin/ros/pycram_ws/src/pycram/resources/objects/spoon.stl


Unknown tag "material" in /robot[@name='milk_object']/link[@name='milk_main']/collision[1]
Unknown tag "material" in /robot[@name='cereal_object']/link[@name='cereal_main']/collision[1]
Unknown tag "material" in /robot[@name='spoon_object']/link[@name='spoon_main']/collision[1]


[INFO] [1740042781.724301]: [cache_manager.py:105:look_for_file_in_data_dir] Found file bowl.stl in /home/hawkin/ros/pycram_ws/src/pycram/resources/objects/bowl.stl


Unknown tag "material" in /robot[@name='bowl_object']/link[@name='bowl_main']/collision[1]


In [4]:

@with_simulated_robot
def move_and_detect(obj_type):
    pick_pose = Pose([2.7, 2.15, 1])

    NavigateAction(target_locations=[Pose([1.7, 2, 0])]).resolve().perform()

    LookAtAction(targets=[pick_pose]).resolve().perform()

    object_desig = DetectAction(technique=DetectionTechnique.TYPES, object_designator_description=BelieveObject(types=[obj_type])).resolve().perform()
    return object_desig[0]


def demo():
    global world, robot, apartment_desig, robot_desig, apartment
    with (simulated_robot):
        ParkArmsAction([Arms.BOTH]).resolve().perform()

        MoveTorsoAction([0.25]).resolve().perform()

        milk_desig = move_and_detect(Milk)

        TransportAction(milk_desig,  [Pose([4.8, 3.55, 0.8])], [Arms.LEFT]).resolve().perform()

        cereal_desig = move_and_detect(Cereal)

        TransportAction(cereal_desig,  [Pose([5.2, 3.4, 0.8], [0, 0, 1, 1])],[Arms.RIGHT]).resolve().perform()

        bowl_desig = move_and_detect(Bowl)

        TransportAction(bowl_desig,  [Pose([5, 3.3, 0.8], [0, 0, 1, 1])], [Arms.LEFT]).resolve().perform()

        # Finding and navigating to the drawer holding the spoon
        handle_desig = ObjectPart(names=["handle_cab10_t"], part_of=apartment_desig.resolve())
        drawer_open_loc = AccessingLocation(handle_desig=handle_desig.resolve(),
                                            robot_desig=robot_desig.resolve()).resolve()

        NavigateAction([drawer_open_loc.pose]).resolve().perform()

        OpenAction(object_designator_description=handle_desig, arms=[drawer_open_loc.arms[0]]).resolve().perform()
        #spoon.detach(apartment)

        # Detect and pickup the spoon
        LookAtAction([apartment.get_link_pose("handle_cab10_t")]).resolve().perform()

        spoon_desigs = DetectAction(technique=DetectionTechnique.TYPES,
                                   object_designator_description=BelieveObject(types=[Spoon])).resolve().perform()
        spoon_desig = spoon_desigs[0]
        spoon = world.get_object_by_type(Spoon)[0]
        spoon.detach(apartment)
        #spoon_desig.detach(apartment)
        pickup_arm = Arms.LEFT if drawer_open_loc.arms[0] == Arms.RIGHT else Arms.RIGHT
        PickUpAction(spoon_desig, [pickup_arm], [Grasp.TOP]).resolve().perform()

        ParkArmsAction([Arms.LEFT if pickup_arm == Arms.LEFT else Arms.RIGHT]).resolve().perform()

        CloseAction(object_designator_description=handle_desig, arms=[drawer_open_loc.arms[0]]).resolve().perform()

        ParkArmsAction([Arms.BOTH]).resolve().perform()

        MoveTorsoAction([0.15]).resolve().perform()

        # Find a pose to place the spoon, move and then place it
        spoon_target_pose = Pose([4.85, 3.3, 0.8], [0, 0, 1, 1])
        placing_loc = CostmapLocation(target=spoon_target_pose, reachable_for=robot_desig.resolve()).resolve()

        NavigateAction([placing_loc.pose]).resolve().perform()

        PlaceAction(spoon_desig, [spoon_target_pose], [pickup_arm]).resolve().perform()

        ParkArmsAction([Arms.BOTH]).resolve().perform()
        
demo()

Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]
Unknown tag "rgba_color" in /robot[@name='milk_object']/link[@name='milk_main']/visual[1]/material[@name='white']
Unknown tag "rgba_color" in /robot[@name='cereal_object']/link[@name='cereal_main']/visual[1]/material[@name='white']
Unknown tag "rgba_color" in /robot[@name='spoon_object']/link[@name='spoon_main']/visual[1]/material[@name='white']
Unknown tag "rgba_color" in /robot[@name='bowl_object']/link[@name='bowl_main']/visual[1]/material[@name='white']


[INFO] [1736790939.005102]: [ik.py:79:call_ik] Waiting for IK service: /pr2_left_arm_kinematics/get_ik
[INFO] [1736790939.005975]: Waiting for service: /pr2_left_arm_kinematics/get_ik


In [7]:

def reset():
    global world, apartment
    world.get_object_by_type(Milk)[0].set_pose(pose=Pose([2.5, 2, 1.02]))
    world.get_object_by_type(Cereal)[0].set_pose(pose=Pose([2.5, 2.3, 1.05]))
    spoon = world.get_object_by_type(Spoon)[0]
    spoon.set_pose(pose=Pose([2.4, 2.24, 0.85]))
    apartment.attach(spoon, 'cabinet10_drawer_top')
    world.get_object_by_type(Bowl)[0].set_pose(pose=Pose([2.5, 2.2, 1.02]))
    
reset()

In [13]:
from anytree import RenderTree
import pycram.ros_utils.tf_broadcaster as tf

#@with_tree
def mini():
    with (simulated_robot):
        tf.TFBroadcaster(odom_frame="map", interval=0.1)
        tt = TaskTree()
        action = NavigateAction(target_locations=[Pose([1.3, 3, 0])]).resolve()
        #print(action.__dict__)
        #print("---")
        #print(action.resolve().__dict__)
        #print("---")
        print(action.perform())
        
        print("---")
        print(f"id: {id(action)}")
        print(f"+children: {tt.root.children}")
        #print(f"+get current performable: {tt.current_node.children[0]}")
        found_action = tt.find_node_by_action(action)
        print("---")
        print(f"found action: {found_action}")
        print(RenderTree(tt.root))
        print("---")
        print(f"found {vars(found_action.__dict__.get('action'))}")
        # how to access the name of an action via task tree. Not really pretty, but oh well...
        print(f"found {found_action.__dict__.get('action').__class__.__name__}")
        print("+++")
        print(f"{found_action.__dict__.get('action').target_location.pose}")
        print(f"{found_action.__dict__.get('action').target_location}")
        
        tt.reset_tree()
    
mini()

[INFO] [1737566390.610146]: [NEEM] Initializing object of class NavigateAction
[INFO] [1737566390.620995]: [NEEM] NavigateAction detected
[INFO] [1737566390.675148]: Adding triple: http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#Action_VMCXYSUE, soma:hasGoal, soma:Location_FISKWOGC
[INFO] [1737566390.679244]: [NEEM] processing attribute: exceptions + {} 
[INFO] [1737566390.679851]: [NEEM] processing attribute: state + None 
[INFO] [1737566390.680330]: [NEEM] processing attribute: executing_thread + {} 
[INFO] [1737566390.680753]: [NEEM] processing attribute: threads + [] 
[INFO] [1737566390.681350]: [NEEM] processing attribute: interrupted + False 
[INFO] [1737566390.681999]: [NEEM] processing attribute: name + NavigateAction 
[INFO] [1737566390.682522]: [NEEM] processing attribute: knowledge_condition + <pycram.datastructures.property.SpaceIsFreeProperty object at 0x7fe32a03d640> 
[INFO] [1737566390.683049]: [NEEM] processing attribute: ground + <bound method ActionDesignatorDes

AttributeError: 'NoneType' object has no attribute '__dict__'

In [4]:
# knowrob stuff!
from neem_interface_python import neem_generation as neem
from neem_interface_python import rosprolog_client
import rospy
import importlib


def test_neem():
    #importlib.reload(neem)
    neem.initialized = False
    neem.parent_action = ""
    neem.current_action = None
    with (simulated_robot):
        #NavigateAction = neem.enable_neem_generation()
        
        neem.initialize_neem()
        rospy.sleep(2)
        action = NavigateAction(target_locations=[Pose([1.4, 3, 0])])
        print("-----------")
        resolved = action.resolve()
        resolved.perform()
        print("-----------")
        #rospy.sleep(2)
        #sec = NavigateAction(target_locations=[Pose([1.7, 3, 0])]).resolve()
        #sec.perform()
        print("done")
        rospy.sleep(2)
        neem.stop_episode(neem.neem_output_path)

def test_query():
    know = rosprolog_client.Prolog()
    #query = f"add_subaction_with_task('http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#Action_ZBRTKSFH', 'http://www.ease-crc.org/ont/SOMA.owl#Navigating', SubAction)."
    query = f"instance_of(X,Y)."
    print(know.all_solutions(query))
    

#test_neem()
#test_query()

In [4]:
from neem_interface_python.utils.utils import get_classes_and_parameters_from_file, get_classes_from_file, autogenerate_class_name_to_class
import pycram.designators.action_designator as action_designator
import pycram.designators.location_designator as location_designator
import pycram.designators.object_designator as object_designator
import pycram.designators.motion_designator as motion_designator


#get_classes_from_file(action_designator)
#autogenerate_class_name_to_class("pycram/sry/pycram/designators/location_designator.py")
get_classes_and_parameters_from_file(action_designator)

[('ActionAbstract', []),
 ('CloseAction',
  [('object_designator_description',
    "str(object='') -> str\nstr(bytes_or_buffer[, encoding[, errors]]) -> str\n\nCreate a new string object from the given object. If encoding or\nerrors is specified, then the object must expose a data buffer\nthat will be decoded using the given encoding and error handler.\nOtherwise, returns the result of object.__str__() (if defined)\nor repr(object).\nencoding defaults to sys.getdefaultencoding().\nerrors defaults to 'strict'."),
   ('arms',
    "str(object='') -> str\nstr(bytes_or_buffer[, encoding[, errors]]) -> str\n\nCreate a new string object from the given object. If encoding or\nerrors is specified, then the object must expose a data buffer\nthat will be decoded using the given encoding and error handler.\nOtherwise, returns the result of object.__str__() (if defined)\nor repr(object).\nencoding defaults to sys.getdefaultencoding().\nerrors defaults to 'strict'."),
   ('grasping_prepose_distance'

In [8]:
from neem_interface_python import designator_decorators 
import importlib
importlib.reload(designator_decorators)

def test_new_neem():
    init_neem()
    with (simulated_robot):
        action = NavigateAction(target_locations=[Pose([1.4, 3, 0])])
        print("-----------")
        resolved = action.resolve()
        
        print("-----------")
        resolved.perform()
        
        print("done")
    
test_new_neem()

[INFO] [1740043210.163567]: [KnowRob] initialize client...
[INFO] [1740043210.166107]: [KnowRob]  done.
[INFO] [1740043210.167244]: [NEEM] Initializing connection...
path: /home/hawkin/ros/pycram_ws/src/pycram/src/neem_interface_python/src/neem-interface/src/neem-interface.pl
Query Once: ensure_loaded('/home/hawkin/ros/pycram_ws/src/pycram/src/neem_interface_python/src/neem-interface/src/neem-interface.pl').
[INFO] [1740043210.170435]: [NEEM] Root Action: .
Query Once: mem_episode_start(Action, 'package://iai_apartment/owl/iai-apartment.owl', 'http://knowrob.org/kb/iai-apartment.owl#apartment_root', 'package://iai_apartment/urdf/apartment.urdf','iai_apartment/','package://knowrob/owl/robots/PR2.owl', 'http://knowrob.org/kb/PR2.owl#PR2_0', 'package://knowrob/urdf/pr2.urdf'),ros_logger_start.
root_action: http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#Action_EOZSPCLN
[INFO] [1740043211.043273]: [NEEM] Connection established.
[INFO] [1740043211.044453]: [NEEM] Logging NavigateActio

TypeError: 'bool' object is not subscriptable